# 03 · Gold layer – Nordvia Retail (SAP FI)

**Goal.** Reshape the silver tables into one table per class of the ontology (`ontology/nordvia-finance.ttl`), so that the mappings are a plain column-to-property translation with no logic in them.

**What gold does**
1. Renames the SAP fields to business names and builds the identifier of each entity.
2. Decides the class of each row (for example *Vendor* or *Employee*, *Customer Invoice* or *Reversal Document*).
3. Applies the decisions taken on the data quality log (section 1).
4. Calculates the values the source does not have: monthly exchange rates to euros, the amount in euros and the due date.

**Reading the columns.** Columns ending in `_id`, `_code` or `_number` identify an entity or point to one in another gold table. Columns ending in `_class` or named `side`, `account_type`, `method`, `category` hold the name of a class or individual of the ontology. Columns ending in `_iri` hold a complete IRI.

## 0 · Environment

In [1]:
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

In [2]:
import os, sys
import pandas as pd

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)

print("Python", sys.version.split()[0], "| pandas", pd.__version__)

SILVER_PATH, GOLD_PATH = "silver", "gold"
TABLES = ["T001", "SKA1", "SKAT", "SKB1", "CSKS", "KNA1", "LFA1", "T003", "TBSL", "BKPF", "BSEG"]
S = {t: pd.read_csv(f"{SILVER_PATH}/{t}.csv", dtype=str, keep_default_na=False, encoding="utf-8") for t in TABLES}

def flag(s):
    # silver writes booleans as the text True / False
    return s == "True"

gold = {}

Python 3.12.7 | pandas 3.1.1


## 1 · Decisions applied in gold
Each line is a decision taken on an entry of the data quality log. Entries not listed need nothing here: they are either visible through a class of the ontology or handled in the queries.

| Log | Decision | Where |
|---|---|---|
| DQ-01 | The customer without master record is kept, as an *Unregistered Customer* | 3.5 |
| DQ-03 | The texts of the account that is not in the chart are not loaded | 3.2 |
| DQ-05 | The account without Spanish text gets its English name only | 3.2 |
| DQ-06 | Customer country in upper case | silver |
| DQ-07, DQ-10 | One-time customers and employees get their own class | 3.5, 3.6 |
| DQ-09 | The two records of the same vendor become **one payee with two identifiers**; their line items point to it | 3.6, 4.2 |
| DQ-12 | A document is identified by company, number and fiscal year, never by its reference | 4.1 |
| DQ-14, DQ-15 | Reversals, reversed documents and opening balances are loaded with their own class; the queries leave them out where needed | 4.1 |
| DQ-16 | The local amount is taken from `DMBTR` and never recomputed | 4.2 |
| DQ-17 | Lines without a cost center are loaded **without a cost center**; no dummy cost center is created | 4.2 |
| DQ-19 | The due date of an item uses the terms on its line | 4.2 |
| DQ-20 | Each credit memo is linked to the invoice it relates to | 4.1 |
| DQ-21 | Monthly exchange rates to euros are derived from the documents, with the method recorded | 2 |

The quality flags that have no class in the ontology (for example the probable duplicate invoice, DQ-11) stay in silver and are not taken to the graph.

In [3]:
def date_time(s):
    """2025-06-30 -> 2025-06-30T00:00:00 (the ontology types dates as xsd:dateTime); empty stays empty."""
    return (s + "T00:00:00").where(s != "", "")

def boolean(s):
    """True / False -> true / false, as xsd:boolean is written."""
    return s.map({"True": "true", "False": "false"})

FIBO = "https://spec.edmcouncil.org/fibo/ontology/FND/Accounting/ISO4217-CurrencyCodes/"
CURRENCY_IRI = {"EUR": FIBO + "Euro", "USD": FIBO + "USDollar", "GBP": FIBO + "PoundSterling"}
GROUP_CURRENCY = "EUR"

def doc_id(company, number, year):
    return company + "-" + number + "-" + year

## 2 · Exchange rates to euros
The challenge gives no rate table and asks to derive the rates from the data and explain the method.

**What the data offers.** Every document header carries `KURSF`, the rate from its document currency to the local currency of its company. A document posted in a foreign currency therefore quotes one rate between two currencies on its posting date.

**Method.** One rate per local currency and fiscal period, from that currency to euros:
1. **Direct rate** – average `KURSF` of the documents in that currency booked by the company that keeps its books in euros.
2. **Inverse rate** – if there is none, one divided by the average `KURSF` of the euro documents booked by the company that keeps its books in that currency.
3. **Cross rate** – if there is neither, the rate through the third currency: (third currency → euro) ÷ (third currency → this currency), both taken from the same period.

A monthly average is used because the rate of a pair moves by less than 0.7 % within a month, and the questions are asked per month or per year, not per day.

In [4]:
t001 = S["T001"].set_index("BUKRS")
h = S["BKPF"].assign(local=S["BKPF"]["BUKRS"].map(t001["WAERS"]), rate=pd.to_numeric(S["BKPF"]["KURSF"]))
quotes = h[h["WAERS"] != h["local"]].groupby(["WAERS", "local", "MONAT"])["rate"].agg(["mean", "size"])   
periods = sorted(S["BKPF"]["MONAT"].unique())
currencies = sorted(set(t001["WAERS"]))

def pair_rate(a, b, period):
    if (a, b, period) in quotes.index:
        return quotes.loc[(a, b, period), "mean"], "DirectRate", int(quotes.loc[(a, b, period), "size"])
    if (b, a, period) in quotes.index:
        return 1 / quotes.loc[(b, a, period), "mean"], "InverseRate", int(quotes.loc[(b, a, period), "size"])
    return None

rows = []
for cur in [c for c in currencies if c != GROUP_CURRENCY]:
    for p in periods:
        found = pair_rate(cur, GROUP_CURRENCY, p)
        if found is None:                                   # cross through the third currency
            for third in [c for c in currencies if c not in (cur, GROUP_CURRENCY)]:
                x, y = pair_rate(third, GROUP_CURRENCY, p), pair_rate(third, cur, p)
                if x and y:
                    found = (x[0] / y[0], "CrossRate", x[2] + y[2])
                    break
        assert found is not None, f"no rate for {cur} in period {p}"
        rows.append({"rate_id": f"{cur}-{GROUP_CURRENCY}-2025-{p}", "from_currency": cur, "to_currency": GROUP_CURRENCY,
                     "from_currency_iri": CURRENCY_IRI[cur], "to_currency_iri": CURRENCY_IRI[GROUP_CURRENCY],
                     "fiscal_year": 2025, "fiscal_period": int(p), "rate": round(found[0], 5),
                     "method": found[1], "documents_used": found[2]})
gold["exchange_rate"] = pd.DataFrame(rows)
gold["exchange_rate"].pivot(index="fiscal_period", columns="from_currency", values=["rate", "method", "documents_used"])

rate                method             documents_used    
from_currency      GBP      USD          GBP         USD            GBP USD
fiscal_period                                                              
1              1.18896   0.9692   DirectRate   CrossRate              2   3
2              1.18821  0.96027   DirectRate  DirectRate              1   1
3              1.18623  0.94929  InverseRate  DirectRate              3   3
4              1.18432  0.93592  InverseRate  DirectRate              3   7
5              1.17773  0.92431   DirectRate  DirectRate              1   4
6              1.17594  0.91799   DirectRate  DirectRate              1   2
7              1.17686  0.90626  InverseRate  DirectRate              1   1
8              1.17368  0.89055    CrossRate  DirectRate              3   2
9              1.17165  0.87995   DirectRate  DirectRate              1   2
10             1.17329  0.87268   DirectRate  DirectRate              1   2
11             1.17165   0.8611   DirectRate  DirectRate              4   7
12             1.17004  0.84945  InverseRate  DirectRate              4   2

**Reading.** Most periods have a direct or an inverse quote. A cross rate is needed only where no document links the currency and the euro in that period; `documents_used` shows how many documents each rate rests on, which is the measure of how much to trust it.

## 3 · Master data
### 3.1 Company, chart of accounts and controlling area

In [5]:
d = S["T001"]
gold["company"] = pd.DataFrame({
    "company_code": d["BUKRS"], "legal_name": d["BUTXT"], "city": d["ORT01"], "country": d["LAND1"],
    "local_currency": d["WAERS"], "local_currency_iri": d["WAERS"].map(CURRENCY_IRI),
    "language_tag": d["language_tag"], "chart_of_accounts_code": d["KTOPL"], "fiscal_year_variant": d["PERIV"]})
gold["chart_of_accounts"] = pd.DataFrame({"chart_of_accounts_code": sorted(S["SKA1"]["KTOPL"].unique())})
gold["controlling_area"] = pd.DataFrame({"controlling_area_code": sorted(S["CSKS"]["KOKRS"].unique())})
gold["company"]

,company_code,legal_name,city,country,local_currency,local_currency_iri,language_tag,chart_of_accounts_code,fiscal_year_variant
0,1000,Nordvia Retail Iberia S.L.,Barcelona,ES,EUR,https://spec.edmcouncil.org/fibo/ontology/FND/...,es,NVCA,K4
1,2000,Nordvia Retail US Inc.,Chicago,US,USD,https://spec.edmcouncil.org/fibo/ontology/FND/...,en,NVCA,K4
2,3000,Nordvia Retail UK Ltd.,Manchester,GB,GBP,https://spec.edmcouncil.org/fibo/ontology/FND/...,en,NVCA,K4


### 3.2 G/L accounts
One row per account of the chart. The names in the two languages become four columns, so that each one can be given its language tag in the mapping. Texts of accounts that are not in the chart are left out (DQ-03); an account with no Spanish text simply has those two columns empty (DQ-05).

In [6]:
a, t = S["SKA1"], S["SKAT"][~flag(S["SKAT"]["dq_no_account_definition"])]
names = t.pivot(index="SAKNR", columns="language_tag", values=["TXT50", "TXT20"])
names.columns = [("name_" if f == "TXT50" else "short_name_") + lang for f, lang in names.columns]
g = pd.DataFrame({
    "account_number": a["SAKNR"], "chart_of_accounts_code": a["KTOPL"], "account_group": a["KTOKS"],
    "account_class": flag(a["is_balance_sheet"]).map({True: "BalanceSheetAccount", False: "ProfitAndLossAccount"}),
    "created_on": date_time(a["ERDAT"]), "entered_by": a["ERNAM"]})
gold["gl_account"] = g.merge(names.reset_index().rename(columns={"SAKNR": "account_number"}), on="account_number", how="left")
gold["gl_account"].head()

,account_number,chart_of_accounts_code,account_group,account_class,created_on,entered_by,name_en,name_es,short_name_en,short_name_es
0,0000100000,NVCA,SAKO,BalanceSheetAccount,2019-03-15T00:00:00,MIGRATION,Petty cash,Caja,Petty cash,Caja
1,0000113100,NVCA,SAKO,BalanceSheetAccount,2019-03-15T00:00:00,MIGRATION,Bank - main account,Banco - cuenta principal,Bank - main account,Banco - cuenta princ
2,0000140000,NVCA,SAKO,BalanceSheetAccount,2019-03-15T00:00:00,MIGRATION,Trade receivables,Clientes,Trade receivables,Clientes
3,0000154000,NVCA,SAKO,BalanceSheetAccount,2019-03-15T00:00:00,MIGRATION,Input VAT,IVA soportado,Input VAT,IVA soportado
4,0000160000,NVCA,SAKO,BalanceSheetAccount,2019-03-15T00:00:00,MIGRATION,Trade payables,Proveedores,Trade payables,Proveedores


### 3.3 Company accounts
The set-up of each account in each company code. `account_class` says whether it is a reconciliation account; `extra_class` marks the accounts on which a cost center is mandatory.

In [7]:
d = S["SKB1"]
gold["company_account"] = pd.DataFrame({
    "company_account_id": d["BUKRS"] + "-" + d["SAKNR"], "company_code": d["BUKRS"], "account_number": d["SAKNR"],
    "account_currency_iri": d["WAERS"].map(CURRENCY_IRI),
    "account_class": d["MITKZ"].map({"D": "ReceivablesAccount", "K": "PayablesAccount"}).fillna("CompanyAccount"),
    "extra_class": flag(d["cost_center_required"]).map({True: "CostCenterRequiredAccount", False: ""}),
    "open_item_managed": boolean(d["is_open_item_managed"]), "field_status_group": d["FSTAG"],
    "line_item_display": boolean(d["has_line_item_display"])})
gold["company_account"]["account_class"].value_counts().to_frame("accounts")

,accounts
account_class,
CompanyAccount,74
ReceivablesAccount,3
PayablesAccount,3


### 3.4 Cost centers
`valid_to` keeps `9999-12-31` for cost centers with no end date, as decided.

In [8]:
d = S["CSKS"]
gold["cost_center"] = pd.DataFrame({
    "cost_center_number": d["KOSTL"], "company_code": d["BUKRS"], "controlling_area_code": d["KOKRS"],
    "category": d["KOSAR"].map({"V": "Administration", "H": "Service", "W": "Sales"}),
    "valid_from": date_time(d["DATAB"]), "valid_to": date_time(d["DATBI"]),
    "person_responsible": d["VERAK"], "name": d["LTEXT"], "short_name": d["KTEXT"]})
gold["cost_center"].head()

,cost_center_number,company_code,controlling_area_code,category,valid_from,valid_to,person_responsible,name,short_name
0,0000011010,1000,NV00,Administration,2019-01-01T00:00:00,9999-12-31T00:00:00,M. Puig,Administration,Administration
1,0000011020,1000,NV00,Administration,2019-01-01T00:00:00,9999-12-31T00:00:00,L. Romero,Finance,Finance
2,0000011030,1000,NV00,Service,2019-01-01T00:00:00,9999-12-31T00:00:00,A. Serra,IT,IT
3,0000011040,1000,NV00,Sales,2019-01-01T00:00:00,9999-12-31T00:00:00,C. Vidal,Marketing,Marketing
4,0000011050,1000,NV00,Sales,2019-01-01T00:00:00,9999-12-31T00:00:00,J. Ortega,Sales,Sales


### 3.5 Customers
The class comes from the account group; the customer that silver inferred from the postings (DQ-01) becomes an *Unregistered Customer*. Addresses go to their own table (3.7).

In [9]:
d = S["KNA1"]
cls = d["KTOKD"].map({"KUNA": "DomesticCustomer", "KUNX": "ForeignCustomer", "CPD": "OneTimeCustomer"})
gold["customer"] = pd.DataFrame({
    "customer_number": d["KUNNR"], "customer_class": cls.where(~flag(d["is_inferred"]), "UnregisteredCustomer"),
    "name": d["NAME1"], "tax_id": d["STCD1"], "created_on": date_time(d["ERDAT"]),
    "flagged_for_deletion": boolean(d["is_flagged_for_deletion"]), "payment_terms_code": d["ZTERM"],
    "city": d["ORT01"], "postal_code": d["PSTLZ"], "country": d["LAND1"]})
gold["customer"]["customer_class"].value_counts().to_frame("customers")

,customers
customer_class,
DomesticCustomer,128
ForeignCustomer,25
OneTimeCustomer,3
UnregisteredCustomer,1


### 3.6 Payees: vendors and employees
`LFA1` holds everyone the company pays. Each row becomes a *Vendor* or an *Employee*.

**Duplicate vendor (DQ-09).** The two records that share a Tax ID become one payee, under the older vendor number. Both numbers are kept in `payee_identifier`, so either one still finds the vendor; `payee_key` translates every original number into the number of its payee and is used for the line items in 4.2.

In [10]:
d = S["LFA1"]
payee_key = d.set_index("LIFNR")["canonical_lifnr"]               
keep = d[d["LIFNR"] == d["canonical_lifnr"]]
gold["payee"] = pd.DataFrame({
    "payee_number": keep["LIFNR"], "payee_class": flag(keep["is_employee"]).map({True: "Employee", False: "Vendor"}),
    "name": keep["NAME1"], "tax_id": keep["STCD1"], "created_on": date_time(keep["ERDAT"]),
    "posting_blocked": boolean(keep["is_blocked"]), "payment_terms_code": keep["ZTERM"],
    "city": keep["ORT01"], "postal_code": keep["PSTLZ"], "country": keep["LAND1"]}).reset_index(drop=True)
gold["payee_identifier"] = pd.DataFrame({"payee_number": d["canonical_lifnr"], "identifier": d["LIFNR"]})

merged = gold["payee_identifier"][gold["payee_identifier"].duplicated("payee_number", keep=False)]
print(len(d), "records in LFA1 ->", len(gold["payee"]), "payees")
merged.merge(d[["LIFNR", "NAME1", "STCD1", "ERDAT"]], left_on="identifier", right_on="LIFNR").drop(columns="LIFNR")

72 records in LFA1 -> 71 payees


,payee_number,identifier,NAME1,STCD1,ERDAT
0,0000200004,0000200004,Harbor Systems S.L.U.,A32661340,2019-08-19
1,0000200004,0000200072,HARBOR SYSTEMS SLU,A32661340,2025-03-12


### 3.7 Addresses
One address per company, customer and payee that has any address data. One-time customers have a country and nothing else. An entity with no address data at all (the unregistered customer) gets no address, rather than an empty one.

In [11]:
parts = []
for owner, table, key in [("company", "company", "company_code"), ("customer", "customer", "customer_number"), ("payee", "payee", "payee_number")]:
    t = gold[table]
    a = pd.DataFrame({"address_id": owner + "-" + t[key], "owner_type": owner, "owner_id": t[key], "city": t["city"],
                      "postal_code": t["postal_code"] if "postal_code" in t else "", "country": t["country"]})
    has_data = (a[["city", "postal_code", "country"]] != "").any(axis=1)
    parts.append(a[has_data])
    gold[table] = t.drop(columns=[c for c in ["city", "postal_code", "country"] if c in t]).assign(
        address_id=a["address_id"].where(has_data, ""))
gold["address"] = pd.concat(parts, ignore_index=True)
gold["address"].groupby("owner_type").size().to_frame("addresses")

,addresses
owner_type,
company,3
customer,156
payee,71


### 3.8 Document types and posting keys

In [12]:
d = S["T003"]
gold["document_type"] = pd.DataFrame({"document_type_code": d["BLART"], "name": d["LTEXT"], "number_range": d["NUMKR"]})
d = S["TBSL"]
SIDE = {"S": "Debit", "H": "Credit"}
ACCOUNT_TYPE = {"D": "CustomerAccountType", "K": "PayeeAccountType", "S": "GLAccountType"}
gold["posting_key"] = pd.DataFrame({"posting_key_code": d["BSCHL"], "side": d["SHKZG"].map(SIDE),
                                    "account_type": d["KOART"].map(ACCOUNT_TYPE), "name": d["LTEXT"]})
gold["posting_key"]

,posting_key_code,side,account_type,name
0,01,Debit,CustomerAccountType,Invoice
1,11,Credit,CustomerAccountType,Credit memo
2,15,Credit,CustomerAccountType,Incoming payment
3,21,Debit,PayeeAccountType,Credit memo
4,25,Debit,PayeeAccountType,Outgoing payment
5,31,Credit,PayeeAccountType,Invoice
6,40,Debit,GLAccountType,Debit entry
7,50,Credit,GLAccountType,Credit entry


## 4 · Documents
### 4.1 Accounting documents
- **Identifier:** company + document number + fiscal year (DQ-12).
- **Class:** from the document type; the three migration documents are *Opening Balance Documents* (DQ-15). A document cancelled by a reversal also gets `extra_class = ReversedDocument` (DQ-14).
- **Links between documents:** `reverses_document_id` / `reversed_by_document_id` from the reversal fields, and `credit_for_document_id` from the reference of a credit memo, which holds the number of its invoice (DQ-20).

In [13]:
d = S["BKPF"]
CLASS = {"DR": "CustomerInvoice", "DG": "CustomerCreditMemo", "DZ": "CustomerPayment", "KR": "PayableInvoice",
         "KZ": "OutgoingPayment", "SA": "GLDocument", "AB": "ReversalDocument"}
this = doc_id(d["BUKRS"], d["BELNR"], d["GJAHR"])
partner = doc_id(d["BUKRS"], d["STBLG"], d["STJAH"]).where(d["STBLG"] != "", "")
is_reversal, is_reversed = flag(d["is_reversal_document"]), flag(d["is_reversed"])
invoice = doc_id(d["BUKRS"], d["XBLNR"], d["GJAHR"])                
invoices = set(this[d["BLART"] == "DR"])

gold["accounting_document"] = pd.DataFrame({
    "document_id": this, "company_code": d["BUKRS"], "document_number": d["BELNR"],
    "fiscal_year": d["GJAHR_n"], "fiscal_period": d["MONAT_n"], "document_type_code": d["BLART"],
    "document_class": d["BLART"].map(CLASS).where(~flag(d["is_opening_balance"]), "OpeningBalanceDocument"),
    "extra_class": is_reversed.map({True: "ReversedDocument", False: ""}),
    "document_date": date_time(d["BLDAT"]), "posting_date": date_time(d["BUDAT"]), "entry_date": date_time(d["CPUDT"]),
    "entered_by": d["USNAM"], "document_currency": d["WAERS"], "document_currency_iri": d["WAERS"].map(CURRENCY_IRI),
    "exchange_rate": d["KURSF"], "reference": d["XBLNR"], "header_text": d["BKTXT"],
    "reverses_document_id": partner.where(is_reversal, ""), "reversed_by_document_id": partner.where(is_reversed, ""),
    "credit_for_document_id": invoice.where((d["BLART"] == "DG") & invoice.isin(invoices), "")})
gold["accounting_document"]["document_class"].value_counts().to_frame("documents")

,documents
document_class,
CustomerInvoice,683
CustomerPayment,590
PayableInvoice,481
OutgoingPayment,422
GLDocument,145
CustomerCreditMemo,22
ReversalDocument,7
OpeningBalanceDocument,3


### 4.2 Line items
- **Payee:** the vendor number of the line is translated to its payee, so the lines of the duplicate vendor point to one payee (DQ-09).
- **Cost center:** left empty where the source has none (DQ-17).
- **Amounts:** document and local amounts as in the source, unsigned (DQ-16). `amount_group_currency` is the local amount converted to euros with the rate of its fiscal period from section 2; for the company that keeps its books in euros it is the local amount.
- **Due date:** baseline date plus the net days of the terms on the line (DQ-19). Only customer and payee items that carry terms have one; payments do not.
- **Clearing document:** the source gives only a document number; the document is looked up in the same company, in the year of the clearing date.

In [14]:
d = S["BSEG"].merge(S["BKPF"][["BUKRS", "BELNR", "GJAHR", "MONAT"]], on=["BUKRS", "BELNR", "GJAHR"], how="left")
NET_DAYS = {"0001": 0, "NT15": 15, "NT30": 30, "NT45": 45, "NT60": 60}          

local = d["BUKRS"].map(t001["WAERS"])
rate = gold["exchange_rate"].assign(p=gold["exchange_rate"]["fiscal_period"].map("{:02d}".format)).set_index(["from_currency", "p"])["rate"]
to_euro = pd.Series([1.0 if c == GROUP_CURRENCY else rate[(c, p)] for c, p in zip(local, d["MONAT"])], index=d.index)
dmbtr = pd.to_numeric(d["DMBTR"])

due = (pd.to_datetime(d["ZFBDT"].where(d["ZFBDT"] != "")) + pd.to_timedelta(d["ZTERM"].map(NET_DAYS), unit="D")).dt.strftime("%Y-%m-%d").fillna("")
document = doc_id(d["BUKRS"], d["BELNR"], d["GJAHR"])

gold["line_item"] = pd.DataFrame({
    "line_item_id": document + "-" + d["BUZEI"], "document_id": document, "line_number": d["BUZEI_n"],
    "line_class": d["SHKZG"].map({"S": "DebitLineItem", "H": "CreditLineItem"}),
    "posting_key_code": d["BSCHL"], "side": d["SHKZG"].map(SIDE), "account_type": d["KOART"].map(ACCOUNT_TYPE),
    "account_number": d["HKONT"], "customer_number": d["KUNNR"],
    "payee_number": d["LIFNR"].map(payee_key).fillna(""),
    "amount_document_currency": d["WRBTR"], "amount_local_currency": d["DMBTR"],
    "amount_group_currency": (dmbtr * to_euro).round(2),
    "tax_code": d["MWSKZ"], "cost_center_number": d["KOSTL"], "payment_terms_code": d["ZTERM"],
    "baseline_date": date_time(d["ZFBDT"]), "due_date": date_time(due),
    "cleared_by_document_id": doc_id(d["BUKRS"], d["AUGBL"], d["AUGDT"].str[:4]).where(d["AUGBL"] != "", ""),
    "clearing_date": date_time(d["AUGDT"]), "text": d["SGTXT"]})
gold["line_item"][gold["line_item"]["due_date"] != ""].head()

,line_item_id,document_id,line_number,line_class,posting_key_code,side,account_type,account_number,customer_number,payee_number,amount_document_currency,amount_local_currency,amount_group_currency,tax_code,cost_center_number,payment_terms_code,baseline_date,due_date,cleared_by_document_id,clearing_date,text
315,1000-0200000001-2025-001,1000-0200000001-2025,1,DebitLineItem,21,Debit,PayeeAccountType,0000160000,,0000200016,606.57,606.57,606.57,,,NT60,2025-02-05T00:00:00,2025-04-06T00:00:00,1000-0200000001-2025,2025-02-17T00:00:00,
318,1000-0200000002-2025-001,1000-0200000002-2025,1,CreditLineItem,11,Credit,CustomerAccountType,0000140000,0000100019,,17712.31,17712.31,17712.31,,,0001,2025-08-19T00:00:00,2025-08-19T00:00:00,1000-0200000002-2025,2025-08-22T00:00:00,Sales invoice
1316,1000-1600000001-2025-001,1000-1600000001-2025,1,CreditLineItem,11,Credit,CustomerAccountType,0000140000,0000100042,,555.23,555.23,555.23,,,0001,2025-07-17T00:00:00,2025-07-17T00:00:00,,,Credit memo - returns
1319,1000-1600000002-2025-001,1000-1600000002-2025,1,CreditLineItem,11,Credit,CustomerAccountType,0000140000,0000100025,,2392.06,2392.06,2392.06,,,0001,2025-01-24T00:00:00,2025-01-24T00:00:00,,,Credit memo - returns
1322,1000-1600000003-2025-001,1000-1600000003-2025,1,CreditLineItem,11,Credit,CustomerAccountType,0000140000,0000100042,,730.14,730.14,730.14,,,0001,2025-12-28T00:00:00,2025-12-28T00:00:00,,,Credit memo - returns


## 5 · Checks
### 5.1 Every reference points to an existing row
Each column that points to another gold table is checked against that table. An empty value is allowed (no cost center, no payee); a value that finds nothing stops the notebook.

In [15]:
REFERENCES = [
    ("company", "chart_of_accounts_code", "chart_of_accounts", "chart_of_accounts_code"),
    ("company", "address_id", "address", "address_id"),
    ("gl_account", "chart_of_accounts_code", "chart_of_accounts", "chart_of_accounts_code"),
    ("company_account", "company_code", "company", "company_code"),
    ("company_account", "account_number", "gl_account", "account_number"),
    ("cost_center", "company_code", "company", "company_code"),
    ("cost_center", "controlling_area_code", "controlling_area", "controlling_area_code"),
    ("customer", "address_id", "address", "address_id"),
    ("payee", "address_id", "address", "address_id"),
    ("payee_identifier", "payee_number", "payee", "payee_number"),
    ("accounting_document", "company_code", "company", "company_code"),
    ("accounting_document", "document_type_code", "document_type", "document_type_code"),
    ("accounting_document", "reverses_document_id", "accounting_document", "document_id"),
    ("accounting_document", "reversed_by_document_id", "accounting_document", "document_id"),
    ("accounting_document", "credit_for_document_id", "accounting_document", "document_id"),
    ("line_item", "document_id", "accounting_document", "document_id"),
    ("line_item", "posting_key_code", "posting_key", "posting_key_code"),
    ("line_item", "account_number", "gl_account", "account_number"),
    ("line_item", "customer_number", "customer", "customer_number"),
    ("line_item", "payee_number", "payee", "payee_number"),
    ("line_item", "cost_center_number", "cost_center", "cost_center_number"),
    ("line_item", "cleared_by_document_id", "accounting_document", "document_id"),
]
out = []
for child, col, parent, key in REFERENCES:
    v = gold[child][col].astype(str)
    filled = v[v != ""]
    out.append({"table": child, "column": col, "points_to": parent, "filled": len(filled), "empty": int((v == "").sum()),
                "not_found": int((~filled.isin(gold[parent][key].astype(str))).sum())})
refs = pd.DataFrame(out)
assert refs["not_found"].eq(0).all(), refs[refs["not_found"] > 0]
refs

,table,column,points_to,filled,empty,not_found
0,company,chart_of_accounts_code,chart_of_accounts,3,0,0
1,company,address_id,address,3,0,0
2,gl_account,chart_of_accounts_code,chart_of_accounts,27,0,0
3,company_account,company_code,company,80,0,0
4,company_account,account_number,gl_account,80,0,0
5,cost_center,company_code,company,25,0,0
6,cost_center,controlling_area_code,controlling_area,25,0,0
7,customer,address_id,address,156,1,0
8,payee,address_id,address,71,0,0
9,payee_identifier,payee_number,payee,72,0,0


### 5.2 Counts and totals against silver

In [16]:
li, docs, B = gold["line_item"], gold["accounting_document"], S["BSEG"]
checks = {
    "documents": (len(docs), len(S["BKPF"])),
    "line items": (len(li), len(B)),
    "identifiers are unique": (int(docs["document_id"].is_unique and li["line_item_id"].is_unique and gold["payee"]["payee_number"].is_unique), 1),
    "customers (including the unregistered one)": (len(gold["customer"]), len(S["KNA1"])),
    "payees = vendor records minus merged duplicates": (len(gold["payee"]), S["LFA1"]["canonical_lifnr"].nunique()),
    "every vendor number kept as an identifier": (len(gold["payee_identifier"]), len(S["LFA1"])),
    "G/L accounts": (len(gold["gl_account"]), len(S["SKA1"])),
    "lines with a payee": (int((li["payee_number"] != "").sum()), int((B["LIFNR"] != "").sum())),
    "lines without cost center on accounts that require one": (
        int(((li["cost_center_number"] == "") & flag(B["dq_missing_cost_center"])).sum()), int(flag(B["dq_missing_cost_center"]).sum())),
    "credit memos linked to an invoice": (int((docs["credit_for_document_id"] != "").sum()), int((docs["document_class"] == "CustomerCreditMemo").sum())),
    "reversal links, both directions": (int((docs["reverses_document_id"] != "").sum()), int((docs["reversed_by_document_id"] != "").sum())),
    "open items (no clearing document)": (int(((li["cleared_by_document_id"] == "") & (li["account_type"] != "GLAccountType")).sum()), int(flag(B["is_open_item"]).sum())),
    "exchange rates (2 currencies x 12 periods)": (len(gold["exchange_rate"]), 24),
    "lines with an amount in euros": (int(li["amount_group_currency"].notna().sum()), len(li)),
    "total local amount": (round(pd.to_numeric(li["amount_local_currency"]).sum(), 2), round(pd.to_numeric(B["DMBTR"]).sum(), 2)),
}
result = pd.DataFrame(checks, index=["gold", "expected"]).T
result["ok"] = result["gold"] == result["expected"]
assert result["ok"].all(), result[~result["ok"]]
result

,gold,expected,ok
documents,2353.00,2353.00,True
line items,6062.00,6062.00,True
identifiers are unique,1.00,1.00,True
customers (including the unregistered one),157.00,157.00,True
payees = vendor records minus merged duplicates,71.00,71.00,True
every vendor number kept as an identifier,72.00,72.00,True
G/L accounts,27.00,27.00,True
lines with a payee,906.00,906.00,True
lines without cost center on accounts that require one,5.00,5.00,True
credit memos linked to an invoice,22.00,22.00,True


### 5.3 Every document still balances in euros
Converting each line with one rate per company and period must leave debits equal to credits in every document, apart from rounding to cents.

In [17]:
signed = li["amount_group_currency"].where(li["side"] == "Debit", -li["amount_group_currency"])
balance = signed.groupby(li["document_id"]).sum().round(2)
print("documents:", len(balance), "| largest imbalance in euros:", balance.abs().max())
assert balance.abs().max() <= 0.05

documents: 2352 | largest imbalance in euros: 0.03


## 6 · Save
One CSV per gold table in `gold/`. The last column of the summary names the class of the ontology each table feeds.

In [18]:
FEEDS = {"company": "Company", "chart_of_accounts": "ChartOfAccounts", "controlling_area": "ControllingArea",
         "gl_account": "GLAccount", "company_account": "CompanyAccount", "cost_center": "CostCenter",
         "customer": "Customer", "payee": "Payee (Vendor, Employee)", "payee_identifier": "identifiers of Payee",
         "address": "locn:Address", "document_type": "DocumentType", "posting_key": "PostingKey",
         "accounting_document": "AccountingDocument", "line_item": "LineItem", "exchange_rate": "MonthlyExchangeRate"}
os.makedirs(GOLD_PATH, exist_ok=True)
for name in FEEDS:
    gold[name].to_csv(f"{GOLD_PATH}/{name}.csv", index=False, encoding="utf-8")
assert set(FEEDS) == set(gold)
pd.DataFrame({"rows": {n: len(gold[n]) for n in FEEDS}, "columns": {n: gold[n].shape[1] for n in FEEDS}, "ontology_class": FEEDS})

,rows,columns,ontology_class
company,3,8,Company
chart_of_accounts,1,1,ChartOfAccounts
controlling_area,1,1,ControllingArea
gl_account,27,10,GLAccount
company_account,80,9,CompanyAccount
cost_center,25,9,CostCenter
customer,157,8,Customer
payee,71,8,"Payee (Vendor, Employee)"
payee_identifier,72,2,identifiers of Payee
address,230,6,locn:Address
